In [1]:
import pandas as pd
import json
import os

In [2]:
# Define os caminhos
CSV_AUTORES = "autor_revisado.csv"
CSV_LINKS   = "artigo_link.csv"
JSON_PATH =  "../corpus.json"
JSON_OUT = "../corpus.json"

In [3]:
df_autores = pd.read_csv(CSV_AUTORES, dtype=str).fillna("")
df_links   = pd.read_csv(CSV_LINKS, dtype=str).fillna("")
with open(JSON_PATH, encoding="utf-8") as f:
    corpus = json.load(f)

In [4]:
autores_grouped = df_autores.groupby("key_file")
map_links = dict(zip(df_links["key_file"].str.strip(), df_links["link"].str.strip()))

In [5]:
for artigo in corpus:
    storage_key = artigo.get("storage_key", "")
    key_file = os.path.splitext(os.path.basename(storage_key))[0]

    if key_file in autores_grouped.groups:
        subset = autores_grouped.get_group(key_file)

        # Atualiza título revisado
        titulo_revisado = subset["titulo_revisado"].iloc[0].strip()
        artigo["titulo"] = titulo_revisado

        # Atualiza link, se disponível
        if key_file in map_links:
            artigo["informacoes_url"] = map_links[key_file]

        # Atualiza autores
        lista_autores = []
        for _, row in subset.iterrows():
            autor = {
                "nome": row["autor"].strip(),
                "afiliacao": row["afiliação"].strip()
            }
            orcid = row.get("orcid", "").strip()
            if orcid:
                autor["orcid"] = orcid
            lista_autores.append(autor)
        artigo["autores"] = lista_autores

In [6]:
# 2) Processa cada artigo
for artigo in corpus:
    print(f'"titulo":{artigo.get("titulo")}, \
          \n "informacoes_url": {artigo.get("informacoes_url","")}, \
          \n "autores":')
    for autor in artigo.get("autores",""):
        print("\t",autor)
    print()

"titulo":Multipurpose Quantum Network Simulators: A comparative study,           
 "informacoes_url": https://sol.sbc.org.br/index.php/wqunets/article/view/30010,           
 "autores":
	 {'nome': 'Diego Abreu', 'afiliacao': 'Federal University of Pará (UFPA)', 'orcid': 'https://orcid.org/0000-0003-3969-8322'}
	 {'nome': 'Arthur Pimentel', 'afiliacao': 'Federal University of Pará (UFPA)'}
	 {'nome': 'Polyana Moraes', 'afiliacao': 'Federal University of Pará (UFPA)'}
	 {'nome': 'David Tavares', 'afiliacao': 'Federal University of Pará (UFPA)'}
	 {'nome': 'Alan Veloso', 'afiliacao': 'Federal University of Pará (UFPA)'}
	 {'nome': 'Antônio Abelém', 'afiliacao': 'Federal University of Pará (UFPA)', 'orcid': 'https://orcid.org/0000-0003-4085-6674'}

"titulo":Análise de desempenho de linguagens de programação e bibliotecas quânticas,           
 "informacoes_url": https://sol.sbc.org.br/index.php/wquantum/article/view/17219,           
 "autores":
	 {'nome': 'Jorge G. Viegas', 'afiliacao': '

In [7]:
with open(JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(corpus, f, ensure_ascii=False, indent=4)